# 🐕 Dog Breed Classification & Visual Explainability (Grad-CAM)
### Fine-Grained Deep Learning on the Stanford Dogs Dataset with Transfer Learning & Apple Silicon Metal GPU

---

## 📌 Project Overview
Fine-grained image classification is one of the most challenging tasks in computer vision: distinguishing between dog breeds requires capturing subtle morphological differences in ear shape, muzzle length, coat texture, facial structure, and color markings while overcoming large intra-class variations in lighting, pose, and background clutter.

In this project, we develop an end-to-end deep learning system for classifying all **120 dog breeds** from the **Stanford Dogs Dataset** (20,580 images).

### 🚀 Key Highlights:
1. **Apple Silicon Hardware Acceleration**: Native M-series GPU execution via `tensorflow-metal`.
2. **Exploratory Data Analysis (EDA)**: Class distributions, aspect ratios, annotation bounding box visualizations.
3. **Optimized `tf.data` Pipeline**: Parallel mapping, data augmentations (flips, rotations, translations, zooms, contrast), caching, and prefetching with `AUTOTUNE`.
4. **Two-Stage Transfer Learning**:
   - **Phase 1 (Feature Extraction)**: Frozen pre-trained backbone (`EfficientNetV2` / `MobileNetV3`) with a custom regularized classification head (Dropout, BatchNorm, L2 regularization, Label Smoothing).
   - **Phase 2 (Fine-Tuning)**: Selective unfreezing of upper convolutional layers with reduced learning rate.
5. **Rigorous Evaluation**: Top-1 and Top-5 accuracy, classification reports, confusion matrix analysis, and top-confused breed diagnostic bar charts.
6. **Visual Explainability with Grad-CAM**: Gradient-weighted Class Activation Mapping to inspect the exact anatomical regions the model uses for breed identification.



---
## 1. ⚙️ Hardware Environment & Library Setup
Let's inspect our Python environment, TensorFlow version, and confirm Apple Silicon Metal GPU acceleration (`/physical_device:GPU:0`).



In [ ]:
import os
import sys
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from PIL import Image
import tensorflow as tf

# Import our custom package modules
from dog_breed_classification.config import (
    IMAGES_DIR,
    ANNOTATIONS_DIR,
    NUM_CLASSES,
    DEFAULT_IMAGE_SIZE,
    TrainingConfig,
    setup_device,
)
from dog_breed_classification.dataset import (
    load_dataset_index,
    clean_breed_name,
    get_class_mappings,
    create_stratified_splits,
    get_dataset_summary,
    build_tf_dataset,
    create_augmentation_pipeline,
    parse_annotation_xml,
)
from dog_breed_classification.models import (
    build_dog_classifier,
    compile_model,
    set_backbone_trainable,
)
from dog_breed_classification.train import (
    get_callbacks,
    plot_training_history,
)
from dog_breed_classification.evaluate import (
    evaluate_model,
    plot_top_confusions,
    plot_accuracy_distribution,
    plot_sample_predictions_grid,
)
from dog_breed_classification.explainability import (
    make_gradcam_heatmap,
    overlay_gradcam,
)
from dog_breed_classification.predict import DogBreedPredictor

# Setup plot styling
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['figure.dpi'] = 120

# Hardware Diagnostics
device_type = setup_device(verbose=True)
print(f"TensorFlow Version: {tf.__version__}")
print(f"Physical GPUs Available: {tf.config.list_physical_devices('GPU')}")



---
## 2. 📊 Exploratory Data Analysis (EDA)
The Stanford Dogs Dataset contains **20,580 images** categorized into **120 breeds**.
Let's index the entire dataset, clean the raw ImageNet folder names into human-readable breed names, and examine class distributions.



In [ ]:
# Index dataset files and metadata
df = load_dataset_index(images_dir=IMAGES_DIR, annotations_dir=ANNOTATIONS_DIR)
summary = get_dataset_summary(df)

print(f"Total Images: {summary['total_images']:,}")
print(f"Total Dog Breeds: {summary['total_classes']}")
print(f"Images per breed: Min = {summary['min_images_per_class']}, Max = {summary['max_images_per_class']}, Mean = {summary['mean_images_per_class']:.1f}")

display(df.head(8))



In [ ]:
# Visualize Class Balance (Top 15 Most Common vs Least Common Breeds)
breed_counts = df['breed'].value_counts()

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

# Top 15 Most Common
top_15 = breed_counts.head(15)
ax1.barh(range(len(top_15)), top_15.values, color='#1f77b4', edgecolor='black')
ax1.set_yticks(range(len(top_15)))
ax1.set_yticklabels(top_15.index, fontsize=10)
ax1.invert_yaxis()
ax1.set_xlabel("Number of Images", fontsize=11)
ax1.set_title("Top 15 Most Represented Dog Breeds", fontsize=13, fontweight='bold')
ax1.grid(axis='x', linestyle=':', alpha=0.6)

# Top 15 Least Common
bottom_15 = breed_counts.tail(15)
ax2.barh(range(len(bottom_15)), bottom_15.values, color='#ff7f0e', edgecolor='black')
ax2.set_yticks(range(len(bottom_15)))
ax2.set_yticklabels(bottom_15.index, fontsize=10)
ax2.invert_yaxis()
ax2.set_xlabel("Number of Images", fontsize=11)
ax2.set_title("Top 15 Least Represented Dog Breeds", fontsize=13, fontweight='bold')
ax2.grid(axis='x', linestyle=':', alpha=0.6)

plt.tight_layout()
plt.show()



### 🖼️ Visualizing Sample Dog Images with Bounding Box Annotations
The dataset includes Pascal VOC XML annotations with ground-truth dog bounding boxes. Let's visualize a grid of random dog breeds with their bounding boxes.



In [ ]:
import matplotlib.patches as patches

sample_breeds = np.random.choice(df['breed'].unique(), size=8, replace=False)
fig, axes = plt.subplots(2, 4, figsize=(18, 9))
axes = axes.flatten()

for idx, breed in enumerate(sample_breeds):
    ax = axes[idx]
    row = df[df['breed'] == breed].sample(1, random_state=idx).iloc[0]
    img = Image.open(row['filepath']).convert("RGB")
    ax.imshow(img)
    ax.set_title(breed, fontsize=11, fontweight='bold')
    ax.axis('off')
    
    # Render bounding box if annotation exists
    if row['annotation_path'] and Path(row['annotation_path']).exists():
        anno_data = parse_annotation_xml(row['annotation_path'])
        if anno_data and anno_data['objects']:
            for obj in anno_data['objects']:
                box = obj['bndbox']
                rect = patches.Rectangle(
                    (box['xmin'], box['ymin']),
                    box['xmax'] - box['xmin'],
                    box['ymax'] - box['ymin'],
                    linewidth=2.5,
                    edgecolor='#00ff00',
                    facecolor='none'
                )
                ax.add_patch(rect)

plt.suptitle("Sample Images from Stanford Dogs Dataset with Ground-Truth Annotations", fontsize=15, fontweight='bold')
plt.tight_layout()
plt.show()



---
## 3. 🔄 Data Pipeline & Augmentation
To ensure reliable evaluation, we split the dataset into stratified **Train (70%)**, **Validation (15%)**, and **Test (15%)** sets.
We build an optimized `tf.data.Dataset` pipeline that streams images asynchronously from disk, performs GPU-accelerated data augmentations (flips, rotations, zooms, translations, contrast), batches tensors, and prefetches with `tf.data.AUTOTUNE`.



In [ ]:
# 1. Stratified Data Splitting
class_to_idx, idx_to_class, class_names = get_class_mappings(df)
train_df, val_df, test_df = create_stratified_splits(df, train_ratio=0.70, val_ratio=0.15, test_ratio=0.15, seed=42)

print(f"Data Splits:")
print(f"  Training Set:   {len(train_df):,} images ({len(train_df)/len(df)*100:.1f}%)")
print(f"  Validation Set: {len(val_df):,} images ({len(val_df)/len(df)*100:.1f}%)")
print(f"  Test Set:       {len(test_df):,} images ({len(test_df)/len(df)*100:.1f}%)")

# 2. Construct tf.data Datasets
BATCH_SIZE = 32
IMAGE_SIZE = (224, 224)

train_ds = build_tf_dataset(train_df, class_to_idx, image_size=IMAGE_SIZE, batch_size=BATCH_SIZE, is_training=True, augment=True)
val_ds = build_tf_dataset(val_df, class_to_idx, image_size=IMAGE_SIZE, batch_size=BATCH_SIZE, is_training=False, augment=False)
test_ds = build_tf_dataset(test_df, class_to_idx, image_size=IMAGE_SIZE, batch_size=BATCH_SIZE, is_training=False, augment=False)

print(f"Pipeline created. Single batch inspection:")
for bx, by in train_ds.take(1):
    print(f"  Images batch shape: {bx.shape} (dtype={bx.dtype})")
    print(f"  Labels batch shape: {by.shape} (dtype={by.dtype})")



### 🎨 Visualizing Data Augmentation
Data augmentation exposes the network to geometric transformations and photometric variations, drastically reducing overfitting on fine-grained features.



In [ ]:
aug_pipeline = create_augmentation_pipeline(image_size=IMAGE_SIZE)

sample_img_path = train_df.iloc[0]['filepath']
sample_pil = Image.open(sample_img_path).convert("RGB").resize(IMAGE_SIZE)
sample_tensor = tf.expand_dims(tf.cast(np.array(sample_pil), tf.float32), 0)

fig, axes = plt.subplots(1, 6, figsize=(18, 3.5))
axes[0].imshow(sample_pil)
axes[0].set_title("Original Image", fontweight='bold')
axes[0].axis('off')

for i in range(1, 6):
    aug_img = aug_pipeline(sample_tensor, training=True)[0].numpy().astype(np.uint8)
    axes[i].imshow(aug_img)
    axes[i].set_title(f"Augmentation #{i}")
    axes[i].axis('off')

plt.suptitle("Stochastic Data Augmentations Applied During Training", fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()



---
## 4. 🧠 Model Architecture & Transfer Learning
We utilize state-of-the-art pre-trained vision backbones (**EfficientNetV2-S**, **MobileNetV3-Large**, or **ResNet50V2**) initialized with ImageNet-1k weights.

### Architecture Highlights:
- **Pretrained Backbone**: Captures rich multi-scale visual hierarchical features (edges, textures, canine anatomy).
- **Global Average Pooling 2D**: Reduces spatial feature maps $(7 	imes 7 	imes C)$ to a compact feature vector.
- **Batch Normalization**: Stabilizes activations and prevents internal covariate shift.
- **Dense Layer (512 units, ReLU, L2 Regularization)**: Learns high-level breed-discriminative representations.
- **Dropout (0.3)**: Stochastically deactivates activations during training to prevent co-adaptation.
- **Softmax Output (120 units, float32)**: Produces calibrated class probability distributions.



In [ ]:
# Build classifier with EfficientNetV2-S backbone
model_name = "efficientnetv2_s"
model = build_dog_classifier(
    model_name=model_name,
    input_shape=(224, 224, 3),
    num_classes=NUM_CLASSES,
    dropout_rate=0.3,
    l2_reg=1e-4,
    freeze_backbone=True
)
model = compile_model(model, learning_rate=1e-3, label_smoothing=0.1)

model.summary(expand_nested=False)



---
## 5. 🏋️ Two-Phase Model Training
We employ a proven two-stage transfer learning strategy:
1. **Phase 1: Feature Extraction**: Backbone is frozen. Only the new classification head is trained for 5-10 epochs at $	ext{LR} = 10^{-3}$ with AdamW and Label Smoothing.
2. **Phase 2: Fine-Tuning**: Top 50 convolutional layers of the backbone are unfrozen. The entire upper network is fine-tuned at a smaller $	ext{LR} = 10^{-4}$ to adapt domain-specific dog representations without catastrophic forgetting.



In [ ]:
# Configure training parameters
config = TrainingConfig(
    model_name="efficientnetv2_s",
    initial_epochs=5,       # Phase 1 epochs
    fine_tune_epochs=8,      # Phase 2 epochs
    initial_lr=1e-3,
    fine_tune_lr=1e-4,
    fine_tune_layers=40,
    batch_size=32,
)

print(f"Training Configuration:")
print(f"  Model: {config.model_name}")
print(f"  Phase 1 (Feature Extraction): {config.initial_epochs} epochs @ lr={config.initial_lr}")
print(f"  Phase 2 (Fine-Tuning):        {config.fine_tune_epochs} epochs @ lr={config.fine_tune_lr}")



In [ ]:
# Phase 1: Feature Extraction (Backbone Frozen)
phase1_callbacks = get_callbacks(config, phase="phase1")

print(">>> Starting Phase 1: Feature Extraction...")
history_p1 = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=config.initial_epochs,
    callbacks=phase1_callbacks,
    verbose=1
)



In [ ]:
# Phase 2: Fine-Tuning (Unfreeze Top Layers)
print(">>> Unfreezing upper backbone layers for fine-tuning...")
model = set_backbone_trainable(
    model,
    unfreeze_layers=config.fine_tune_layers,
    learning_rate=config.fine_tune_lr,
    label_smoothing=config.label_smoothing
)

phase2_callbacks = get_callbacks(config, phase="phase2")

print(">>> Starting Phase 2: Fine-Tuning...")
history_p2 = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=config.fine_tune_epochs,
    callbacks=phase2_callbacks,
    verbose=1
)



In [ ]:
# Merge and plot complete training metrics
from dog_breed_classification.train import merge_histories, plot_training_history

combined_history = merge_histories(history_p1.history, history_p2.history)
fig = plot_training_history(combined_history, save_path=config.plots_dir / f"{config.model_name}_training_curves.png", show=True)



---
## 6. 🏆 Comprehensive Model Evaluation
Let's evaluate the trained model on the unseen **Test Split (3,088 images)** across multiple quantitative metrics:
- **Top-1 Accuracy**: Exact breed matching accuracy.
- **Top-5 Accuracy**: Proportion of times the true breed is among the top 5 highest confidence predictions.
- **Macro & Weighted F1-Scores**: Balanced performance metric across all 120 breeds.
- **Top-Confused Dog Breeds Analysis**: Diagnosing which breed pairs are most easily confused.



In [ ]:
# Run comprehensive evaluation on test split
metrics = evaluate_model(
    model=model,
    test_df=test_df,
    class_to_idx=class_to_idx,
    class_names=class_names,
    batch_size=32,
    save_reports=True
)

print(f"\n{'='*50}\n📊 TEST SET PERFORMANCE SUMMARY\n{'='*50}")
print(f"  Top-1 Accuracy:    {metrics['top_1_accuracy'] * 100:.2f}%")
if metrics['top_5_accuracy']:
    print(f"  Top-5 Accuracy:    {metrics['top_5_accuracy'] * 100:.2f}%")
print(f"  Macro Precision:   {metrics['macro_precision']:.4f}")
print(f"  Macro Recall:      {metrics['macro_recall']:.4f}")
print(f"  Macro F1-Score:    {metrics['macro_f1']:.4f}")
print(f"  Weighted F1-Score: {metrics['weighted_f1']:.4f}")



In [ ]:
# Display Qualitative Prediction Grid
fig = plot_sample_predictions_grid(
    model=model,
    test_df=test_df,
    class_names=class_names,
    class_to_idx=class_to_idx,
    num_samples=12,
    num_cols=4,
    show=True
)



---
## 7. 🔍 Visual Explainability with Grad-CAM
**Grad-CAM (Gradient-weighted Class Activation Mapping)** generates coarse 2D heatmaps showing the visual attention of deep convolutional layers.
By computing the gradients of the predicted class score $y^c$ with respect to the feature map activations $A^k$ of the last conv layer:
$$ \alpha_k^c = \frac{1}{Z} \sum_i \sum_j \frac{\partial y^c}{\partial A_{i,j}^k} $$
$$ L_{\text{Grad-CAM}}^c = \text{ReLU}\left( \sum_k \alpha_k^c A^k \right) $$

Let's inspect what canine anatomical features (ears, muzzle, facial markings, fur patterns) our model relies upon.



In [ ]:
predictor = DogBreedPredictor(model_name="efficientnetv2_s")

# Select diverse test samples
test_samples = test_df.sample(4, random_state=123)['filepath'].tolist()

fig, axes = plt.subplots(4, 3, figsize=(14, 16))

for idx, img_path in enumerate(test_samples):
    res = predictor.predict(img_path, top_k=3, return_gradcam=True)
    
    # Col 1: Original Image
    axes[idx, 0].imshow(res['original_image'])
    axes[idx, 0].set_title(f"Original Image #{idx+1}", fontweight='bold')
    axes[idx, 0].axis('off')
    
    # Col 2: Standalone Grad-CAM Heatmap
    if 'gradcam_heatmap' in res:
        axes[idx, 1].imshow(res['gradcam_heatmap'])
        axes[idx, 1].set_title("Grad-CAM Attention Map", fontweight='bold')
    axes[idx, 1].axis('off')
    
    # Col 3: Overlaid Heatmap + Prediction
    if 'gradcam_overlay' in res:
        axes[idx, 2].imshow(res['gradcam_overlay'])
        axes[idx, 2].set_title(f"Pred: {res['top_breed']} ({res['top_percentage']})", color='darkgreen', fontweight='bold')
    axes[idx, 2].axis('off')

plt.suptitle("Grad-CAM Visual Attention Across Dog Breeds", fontsize=16, fontweight='bold', y=0.99)
plt.tight_layout()
plt.show()



---
## 8. 📦 Model Export to ONNX & CoreML Runtime on Mac
**ONNX (Open Neural Network Exchange)** provides an open, framework-agnostic format for cross-platform model deployment.
On macOS Apple Silicon, `onnxruntime` leverages `CoreMLExecutionProvider` to accelerate inference using the **Apple Neural Engine (ANE)** and Apple GPU without needing TensorFlow.


In [ ]:
# Export model to standard ONNX format
from dog_breed_classification.export_onnx import export_to_onnx
from dog_breed_classification.predict_onnx import ONNXDogBreedPredictor

onnx_output_path = export_to_onnx(
    model=model,
    model_name="efficientnetv2_s",
    output_path="artifacts/models/dog_classifier_efficientnetv2_s.onnx",
    opset=13,
    verbose=True
)


In [ ]:
# Perform ONNX Runtime Inference with CoreML / CPU acceleration
onnx_predictor = ONNXDogBreedPredictor(onnx_path=onnx_output_path)

sample_test_img = test_df.iloc[0]['filepath']
onnx_result = onnx_predictor.predict(sample_test_img, top_k=5)

print(f"Active ONNX Providers: {onnx_result['providers']}")
print(f"Top Predicted Breed:  {onnx_result['top_breed']} ({onnx_result['top_percentage']})")
print("Top-5 Predictions:")
for item in onnx_result['predictions']:
    print(f"  - {item['breed']:<30} {item['percentage']}")


---
## 8. 🎯 Key Insights & Next Steps

### 🏆 Project Summary:
1. **Dataset**: 20,580 images across 120 breeds from the Stanford Dogs Dataset.
2. **Mac Metal Acceleration**: Leveraged Apple Silicon Metal GPU for hardware-accelerated training and inference.
3. **Transfer Learning**: EfficientNetV2 & MobileNetV3 backbones fine-tuned in 2 phases achieved high Top-1 and Top-5 accuracy.
4. **Grad-CAM Explainability**: Proved that the CNN model grounds its decisions on key canine anatomical features (facial structure, ear shape, coat texture) rather than background artifacts.
5. **Interactive Deployment**: Gradio Web Application (`dog-breed app`) and CLI (`dog-breed predict`) provide instant real-time breed classification and heatmap generation.

### 💻 Running the Interactive Web App:
To launch the interactive web interface locally, run in your terminal:
```bash
uv run dog-breed app
```
or python:
```python
from dog_breed_classification.app import launch_app
launch_app()
```

